In [32]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

In [33]:
train_transform = transforms.Compose(
    [transforms.RandomHorizontalFlip(),
     transforms.RandomCrop(32, padding=4),
     transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]
)

train_set = datasets.CIFAR10(root='../data', train=True, download=True, transform=train_transform)
train_loader = DataLoader(train_set, batch_size=8, shuffle=True)

test_transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]
)

test_set = datasets.CIFAR10(root='../data', train=False, download=True, transform=test_transform)
test_loader = DataLoader(test_set, batch_size=10000, shuffle=False)

In [34]:
class LeNet(nn.Module):
    def __init__(self):
        super(LeNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        # x = torch.flatten(x, 1)
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [35]:
leNet = LeNet()
print(leNet)    

LeNet(
  (conv1): Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)


In [36]:
def train(model: nn.Module, device: torch.device, train_loader: DataLoader, optimizer: optim.Optimizer, epoch: int):
    model.train()
    
    loss_total = 0.0
    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        output = model(inputs)
        loss = F.cross_entropy(output, labels)
        loss.backward()
        optimizer.step()
        
        loss_total += loss.item()
        if (i + 1) % 100 == 0:
            print(f'Train Epoch: {epoch} [{(i + 1) * len(inputs)}/{len(train_loader.dataset)} ({100. * (i + 1) / len(train_loader):.0f}%)]\tLoss: {loss_total / 100:.6f}')
            loss_total = 0.0

In [37]:
def test(model: nn.Module, device: torch.device, test_loader: DataLoader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            output = model(inputs)
            test_loss += F.cross_entropy(output, labels, reduction='sum').item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(labels.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    accuracy = 100. * correct / len(test_loader.dataset)
    print(f'\nTest set: Average loss: {test_loss:.4f}, Accuracy: {correct}/{len(test_loader.dataset)} ({accuracy:.0f}%)\n')

In [38]:
optimizer = optim.Adam(leNet.parameters(), lr=0.001)
device = torch.device('cpu')

for epoch in range(5):
    train(leNet, device, train_loader, optimizer, epoch)
    test(leNet, device, test_loader)

Train Epoch: 0 [800/50000 (2%)]	Loss: 2.253834
Train Epoch: 0 [1600/50000 (3%)]	Loss: 2.129372
Train Epoch: 0 [2400/50000 (5%)]	Loss: 2.086508
Train Epoch: 0 [3200/50000 (6%)]	Loss: 2.044954
Train Epoch: 0 [4000/50000 (8%)]	Loss: 2.014849
Train Epoch: 0 [4800/50000 (10%)]	Loss: 1.964799
Train Epoch: 0 [5600/50000 (11%)]	Loss: 1.931708
Train Epoch: 0 [6400/50000 (13%)]	Loss: 1.882072
Train Epoch: 0 [7200/50000 (14%)]	Loss: 1.933526
Train Epoch: 0 [8000/50000 (16%)]	Loss: 1.875801
Train Epoch: 0 [8800/50000 (18%)]	Loss: 1.796338
Train Epoch: 0 [9600/50000 (19%)]	Loss: 1.841054
Train Epoch: 0 [10400/50000 (21%)]	Loss: 1.774783
Train Epoch: 0 [11200/50000 (22%)]	Loss: 1.795409
Train Epoch: 0 [12000/50000 (24%)]	Loss: 1.724761
Train Epoch: 0 [12800/50000 (26%)]	Loss: 1.737046
Train Epoch: 0 [13600/50000 (27%)]	Loss: 1.750562
Train Epoch: 0 [14400/50000 (29%)]	Loss: 1.729876
Train Epoch: 0 [15200/50000 (30%)]	Loss: 1.751755
Train Epoch: 0 [16000/50000 (32%)]	Loss: 1.709000
Train Epoch: 0 [16

In [39]:
model_path = "./state_dicts/lenet_cifar10.pth"
torch.save(leNet.state_dict(), model_path)

In [40]:
leNet_cached = LeNet()
leNet_cached.load_state_dict(torch.load(model_path))

<All keys matched successfully>

In [41]:
leNet_cached

LeNet(
  (conv1): Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)